# PRODUCTION

In [ ]:
from pathlib import Path
import pandas as pd
import re

def read_parquet(path_to_file):
    # Set your parquet file path here (works from project root or pdf_reader/)
    parquet_rel = Path(path_to_file)
    parquet_path = parquet_rel if parquet_rel.exists() else Path("..") / parquet_rel
    parquet_path = parquet_path.resolve()

    # Read parquet into a DataFrame ("unpack")
    df = pd.read_parquet(parquet_path)
    df = df.drop_duplicates()  # Remove duplicates if needed

    # Quick check
    print(f"Using: {parquet_path}")
    print("Unpackaging compleet.")
    print(f"DataFrame shape: {df.shape}")

    return df

df = read_parquet("Data/Exploring_meta/thesis_meta_combined.parquet")

def affiliations_cleaning(df):
    pattern = re.compile(
        r"(Department of [^|]*?, Technical University of Denmark|"
        r"Institut for [^|]*?, Danmarks Tekniske Universitet)"
    )

    def clean_affiliations_cell(value):
        if pd.isna(value):
            return value  # or "" if you prefer empty string

        text = (
            str(value)
            .replace("[affiliation name could not be established]|", "")
            .replace("|[affiliation unknown]", "")
            .strip("|")
        )

        matches = pattern.findall(text)

        cleaned = [
            m.replace(", Technical University of Denmark", "")
            .replace(", Danmarks Tekniske Universitet", "")
            .strip()
            for m in matches
        ]

        cleaned = list(dict.fromkeys([c for c in cleaned if c]))  # dedupe, keep order
        return "|".join(cleaned)

    # 1) Keep original
    df["Affiliations_raw"] = df["Affiliations"]

    # 2) Create cleaned version
    df["Affiliations_clean"] = df["Affiliations_raw"].apply(clean_affiliations_cell)

    # Optional: if you eventually want cleaned to become main column
    # df["Affiliations"] = df["Affiliations_clean"]
    return df

df_aff = affiliations_cleaning(df)
display(df_aff[["Affiliations_raw", "Affiliations_clean"]].head(10))

aff_count = len(df_aff["Affiliations_clean"].dropna().unique())
print(f"Unique cleaned affiliations: {aff_count}")



Using: /Users/oliver/Desktop/MSc_Speciale/Thesis_OCR/Data/Exploring_meta/thesis_meta_combined.parquet
Unpackaging compleet.
DataFrame shape: (13444, 40)


,Affiliations_raw,Affiliations_clean
0,"Depatment of Systems Biology, Technical Univer...",Department of Systems Biology
1,[affiliation name could not be established]|Au...,
2,[affiliation name could not be established]|In...,"Institut for Planlægning, Innovation og Ledelse"
3,"Ørsted•DTU, Technical University of Denmark",
4,"Intelligent Signal Processing, Informatics and...",
5,[affiliation name could not be established]|De...,Department of Transport
6,[affiliation name could not be established]|De...,Department of Informatics and Mathematical Mod...
7,"Materials Research Division, Risø National Lab...",
8,[affiliation name could not be established]|Em...,Department of Informatics and Mathematical Mod...
9,"CHEC Research Centre, Department of Chemical E...",Department of Chemical Engineering


69

## Prod - SUSPENDED

In [ ]:
from pathlib import Path
import pandas as pd
import re

def read_parquet_interact(path_to_file):
    # Set your parquet file path here (works from project root or pdf_reader/)
    parquet_rel = Path(path_to_file)
    parquet_path = parquet_rel if parquet_rel.exists() else Path("..") / parquet_rel
    parquet_path = parquet_path.resolve()

    # Read parquet into a DataFrame ("unpack")
    df = pd.read_parquet(parquet_path)
    df = df.drop_duplicates()  # Remove duplicates if needed

    # Quick check
    print(f"Using: {parquet_path}")
    print("Unpackaging compleet.")
    
    commands = ["1", "yes", "Yes", "YES", "y", "Y"]
    print()
    print("To display 'df' input: 1, yes, Yes, YES, y, Y")
    user = input()

    #if display_command in commands:
    if user in commands:
        print(f"data shape: {df.shape}")
        display(df.head())
    else:
        print(f"data shape: {df.shape}")

    # Optional: export to CSV if needed
    # df.to_csv(parquet_path.with_suffix(".csv"), index=False)

    return df

df = read_parquet_interact("Data/Exploring_meta/thesis_meta_combined.parquet")

def unique_find_affiliations(df):

    unique_affiliations = (
        df['Affiliations']
        #.dropna()
        .str.replace('[affiliation name could not be established]|', '', regex=False)
        .str.replace('|[affiliation unknown]', '', regex=False)
        .str.strip('|')
        .unique()
    )

    commands = ["1", "yes", "Yes", "YES", "y", "Y"]
    print()
    print("To display 'Unique Affiliations' input: 1, yes, Yes, YES, y, Y")
    user = input()

    #if display_command in commands:
    if user in commands:
        print("Unique Affiliations:", len(unique_affiliations))
        print(unique_affiliations)
    else:
        print("Unique Affiliations:", len(unique_affiliations))

    return unique_affiliations

unique_affiliations = unique_find_affiliations(df)
    
def clean_affiliations(unique_affiliations):
    pattern = (
        r"(Department of [^|]*?, Technical University of Denmark|"
        r"Institut for [^|]*?, Danmarks Tekniske Universitet)"
    )

    cleaned_affiliations = (
        pd.Series(unique_affiliations)
        .str.findall(pattern)
        .explode()
        #.dropna()
        .drop_duplicates()
        .str.replace(', Technical University of Denmark', '', regex=False)
        .str.replace(', Danmarks Tekniske Universitet', '', regex=False)
        .to_numpy()
    )

    print("Cleaned Affiliations:", len(cleaned_affiliations))
    print(cleaned_affiliations)

    commands = ["1", "yes", "Yes", "YES", "y", "Y"]
    print()
    print("To display 'Cleaned Affiliations' input: 1, yes, Yes, YES, y, Y")
    user = input()

    #if display_command in commands:
    if user in commands:
        print("Cleaned Affiliations:", len(cleaned_affiliations))
        print(cleaned_affiliations)
    else:
        print("Cleaned Affiliations:", len(cleaned_affiliations))

    return cleaned_affiliations

cleaned_affiliations = clean_affiliations(unique_affiliations)

def filter_affiliations(df):
    unique_aff = unique_find_affiliations(df)
    filtered_df = clean_affiliations(unique_aff)
    print("Filtered DataFrame shape:", filtered_df.shape)
    display(filtered_df.head())
    return filtered_df

Using: /Users/oliver/Desktop/MSc_Speciale/Thesis_OCR/Data/Exploring_meta/thesis_meta_combined.parquet
Unpackaging compleet.

To display 'df' input: 1, yes, Yes, YES, y, Y
data shape: (19690, 40)

To display 'Unique Affiliations' input: 1, yes, Yes, YES, y, Y
Unique Affiliations: 463
Cleaned Affiliations: 36
['Department of Systems Biology' nan
 'Institut for Planlægning, Innovation og Ledelse'
 'Department of Transport'
 'Department of Informatics and Mathematical Modeling'
 'Department of Chemical Engineering'
 'Department of Chemical and Biochemical Engineering'
 'Department of Electrical Engineering'
 'Institut for Kommunikation, Optik & Materialer'
 'Department of Micro and Nanotechnology'
 'Department of Manufacturing Engineering and Management'
 'Institut for Informatik og Matematisk Modellering'
 'Department of Communications, Optics & Materials'
 'Department of Mechanical Engineering'
 'Department of Management Engineering' 'Institut for Mekanisk Teknologi'
 'Institut for Kemit

# EXPLORATION

In [9]:
display(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 19690 entries, 0 to 19689
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   abstract_ts                   1073 non-null   str    
 1   access_ss                     19690 non-null  str    
 2   Affiliations                  2876 non-null   str    
 3   Timestamp                     19690 non-null  str    
 4   Author                        19682 non-null  str    
 5   citation_count_i              0 non-null      float64
 6   ID                            19690 non-null  int64  
 7   dtu_library_collection_facet  0 non-null      float64
 8   collection_facet              17783 non-null  str    
 9   Publication Year              19690 non-null  int64  
 10  Conference                    0 non-null      float64
 11  DOI                           0 non-null      float64
 12  Editor                        0 non-null      float64
 13  embargo_ssf 

None

# DEVELOPMENT

to match .parquet med abstract .json:
* Key 1: 'member_id_ss' (fra parquet) == 'filename' before '_' (fra .json)
* Key 2: 'Title' (fra .parquet) == 'title' (fra .json)

In [ ]:
def replace_affiliation_names(df):
    unique_affiliations = (
        df['Affiliations']
        #.dropna()
        .str.replace('[affiliation name could not be established]|', '', regex=False)
        .str.replace('|[affiliation unknown]', '', regex=False)
        .str.strip('|')
        .unique()
    )
    
    pattern = (
        r"(Department of [^|]*?, Technical University of Denmark|"
        r"Institut for [^|]*?, Danmarks Tekniske Universitet)"
    )

    cleaned_affiliations = (
        pd.Series(unique_affiliations)
        .str.findall(pattern)
        .explode()
        #.dropna()
        .drop_duplicates()
        .str.replace(', Technical University of Denmark', '', regex=False)
        .str.replace(', Danmarks Tekniske Universitet', '', regex=False)
        .to_numpy()
    )
    return cleaned_affiliations

#df.iloc[0]['Affiliations'].type

replace_affiliation_names(df.iloc[0])

AttributeError: 'str' object has no attribute 'str'

In [29]:
import re
import pandas as pd

pattern = re.compile(
    r"(Department of [^|]*?, Technical University of Denmark|"
    r"Institut for [^|]*?, Danmarks Tekniske Universitet)"
)

def clean_affiliations_cell(value):
    if pd.isna(value):
        return value  # or "" if you prefer empty string

    text = (
        str(value)
        .replace("[affiliation name could not be established]|", "")
        .replace("|[affiliation unknown]", "")
        .strip("|")
    )

    matches = pattern.findall(text)

    cleaned = [
        m.replace(", Technical University of Denmark", "")
         .replace(", Danmarks Tekniske Universitet", "")
         .strip()
        for m in matches
    ]

    cleaned = list(dict.fromkeys([c for c in cleaned if c]))  # dedupe, keep order
    return "|".join(cleaned)

# 1) Keep original
df["Affiliations_raw"] = df["Affiliations"]

# 2) Create cleaned version
df["Affiliations_clean"] = df["Affiliations_raw"].apply(clean_affiliations_cell)

# Optional: if you eventually want cleaned to become main column
# df["Affiliations"] = df["Affiliations_clean"]

In [33]:
display(df[["Affiliations_raw", "Affiliations_clean"]].head(10))
df["Affiliations_raw"][1]

,Affiliations_raw,Affiliations_clean
0,"Depatment of Systems Biology, Technical Univer...",Department of Systems Biology
1,[affiliation name could not be established]|Au...,
2,[affiliation name could not be established]|In...,"Institut for Planlægning, Innovation og Ledelse"
3,"Ørsted•DTU, Technical University of Denmark",
4,"Intelligent Signal Processing, Informatics and...",
5,[affiliation name could not be established]|De...,Department of Transport
6,[affiliation name could not be established]|De...,Department of Informatics and Mathematical Mod...
7,"Materials Research Division, Risø National Lab...",
8,[affiliation name could not be established]|Em...,Department of Informatics and Mathematical Mod...
9,"CHEC Research Centre, Department of Chemical E...",Department of Chemical Engineering


'[affiliation name could not be established]|Automation, Ørsted DTU, Technical University of Denmark|[affiliation unknown]'

In [ ]:
df[df['Publisher'].notna()]['Publisher']

In [ ]:
unique_publishers = df['Publisher'].dropna().unique()
print("Unique Publishers:", len(unique_publishers))
print(unique_publishers)